In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, plotly.express as px
from src.data.loading import load_processed
from src.features.build import make_features
from src.models.anomaly import detect, explain, to_events

TRAIN_END = pd.Timestamp("2019-03-01")
ROOT = "../data/processed"

def run(h, hourly=None):
    hourly = load_processed(h, "hourly", ROOT) if hourly is None else hourly
    return detect(hourly, make_features(hourly, h), TRAIN_END)

In [2]:
rows = []
for h in [17, 36, 29, 16, 5]:
    res = run(h)
    tr, te = res[res.index < TRAIN_END], res[res.index >= TRAIN_END]
    rows.append({"house_id": h,
                 "flag_pct_train": 100 * tr.flag.mean(),
                 "flag_pct_test": 100 * te.flag.mean(),
                 "flags_per_week_test": te.flag.sum() / (len(te) / 168)})
pd.DataFrame(rows).round(2)

,house_id,flag_pct_train,flag_pct_test,flags_per_week_test
0,17,0.98,0.45,0.76
1,36,0.99,1.09,1.83
2,29,0.99,1.81,3.04
3,16,0.99,1.68,2.82
4,5,0.98,0.59,0.99


In [3]:
def inject_test(h, n_nights=12, seed=42):
    hourly = load_processed(h, "hourly", ROOT)
    rng = np.random.default_rng(seed)
    spike = float(hourly["kwh"].quantile(0.95))     # a heavy load, like a faulty appliance
    nights = rng.choice(pd.date_range("2019-03-05", "2019-05-28", freq="D"),
                        n_nights, replace=False)
    hours_hit = [pd.Timestamp(n) + pd.Timedelta(hours=k) for n in nights for k in (2, 3, 4)]

    inj = hourly.copy()
    inj.loc[hours_hit, "kwh"] = inj.loc[hours_hit, "kwh"] + spike
    res = run(h, inj)

    caught = res.loc[hours_hit, "flag"].to_numpy().reshape(n_nights, 3)
    test_part = res[res.index >= TRAIN_END]
    other = int(test_part.flag.sum() - caught.sum())
    return {"house_id": h, "spike_kwh": round(spike, 2),
            "nights_caught_pct": 100 * caught.any(axis=1).mean(),
            "hours_caught_pct": 100 * caught.mean(),
            "other_flags_per_week": other / (len(test_part) / 168)}

pd.DataFrame([inject_test(h) for h in [17, 36, 29, 16, 5]]).round(1)

,house_id,spike_kwh,nights_caught_pct,hours_caught_pct,other_flags_per_week
0,17,1.3,91.7,86.1,0.2
1,36,1.8,100.0,100.0,1.2
2,29,3.4,83.3,69.4,1.8
3,16,5.0,100.0,94.4,2.4
4,5,2.0,91.7,88.9,0.6


In [4]:
res = run(17)
f = explain(res[res.index >= TRAIN_END])
ev = to_events(f)
print(len(f), "flagged hours ->", len(ev), "alerts")
display(f.sort_values("score", ascending=False)[["reason"]].head(8))

te = res[res.index >= TRAIN_END]
fig = px.line(te["kwh"], title="House 17: hourly kWh with flagged hours")
fig.add_scatter(x=te.index[te.flag], y=te.kwh[te.flag], mode="markers", name="flagged")
fig.show()

10 flagged hours -> 7 alerts


,reason
timestamp,
2019-04-06 11:00:00,"11:00 usage 0.43 kWh vs usual 0.11 kWh (3.5x, ..."
2019-04-28 23:00:00,"23:00 usage 1.25 kWh vs usual 0.48 kWh (2.5x, ..."
2019-04-29 22:00:00,"22:00 usage 1.41 kWh vs usual 0.32 kWh (4.2x, ..."
2019-04-29 00:00:00,"00:00 usage 1.26 kWh vs usual 0.48 kWh (2.6x, ..."
2019-04-29 23:00:00,"23:00 usage 1.46 kWh vs usual 0.62 kWh (2.3x, ..."
2019-04-21 19:00:00,"19:00 usage 0.62 kWh vs usual 0.22 kWh (2.6x, ..."
2019-04-21 18:00:00,"18:00 usage 0.58 kWh vs usual 0.20 kWh (2.7x, ..."
2019-04-27 16:00:00,"16:00 usage 0.61 kWh vs usual 0.18 kWh (3.1x, ..."
